# Fine-Tuning a Code LLM with LoRA — Step by Step

This notebook walks through every stage of fine-tuning **Qwen2.5-Coder-7B-Instruct** on our C optimization dataset using **QLoRA** (Quantized Low-Rank Adaptation).

By the end you will understand:
- What LoRA actually does to a model's weights
- Why quantization (QLoRA) lets you train 7B models on a single GPU
- How to format training data as conversations
- How to run a training loop and read the loss curve
- How to save, merge, and serve the fine-tuned model

**Hardware needed:** a GPU with ≥16 GB VRAM (e.g. RTX 3090/4090, A10, A100).  
For experimentation on less VRAM (8–12 GB), use `load_in_4bit=True` and `max_seq_length=1024`.

---

## Table of contents
0. [Colab setup (path + repo)](#0-setup)
1. [Background — what is LoRA?](#1-background)
2. [Install dependencies](#2-install)
3. [Understand the training data](#3-data)
4. [Load the base model](#4-model)
5. [Apply LoRA adapters](#5-lora)
6. [Format data for training](#6-format)
7. [Train](#7-train)
8. [Inspect the loss curve](#8-loss)
9. [Benchmark on the full dataset](#9-benchmark)
10. [Save and serve](#10-save)

## 0. Colab setup

**Run this cell first.** It clones the benchmark repo (if not already present) and sets the working directory so every subsequent cell resolves paths correctly.

If you already cloned the repo to a different location, change `REPO_DIR` below.

In [ ]:
import os, sys
from pathlib import Path

# ── Configure this if your clone is somewhere else ────────────────────────────
REPO_DIR = "/content/benchmark"
REPO_URL = "https://github.com/wlu03/pattern-driven-optimization-benchmark"
# ─────────────────────────────────────────────────────────────────────────────

REPO_ROOT = Path(REPO_DIR)

# Clone the repo if it isn't there yet
if not REPO_ROOT.exists():
    print(f"Cloning repo → {REPO_ROOT}")
    os.system(f"git clone {REPO_URL} {REPO_ROOT}")
else:
    print(f"Repo found at {REPO_ROOT}")

# Change into fine_tune/ so subprocess calls (prepare_finetune_data.py etc.) work
os.chdir(REPO_ROOT / "fine_tune")
print(f"Working directory: {os.getcwd()}")

# Add repo root to Python path so compiler.py / patterns.py / prompts.py are importable
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"sys.path includes: {REPO_ROOT}")

assert (REPO_ROOT / "compiler.py").exists(), \
    f"compiler.py not found in {REPO_ROOT} — check REPO_DIR above"
print("Setup complete.")

## 1. Background — what is LoRA?

A 7B parameter model has ~7 billion floating-point numbers. Full fine-tuning updates **all** of them, which requires storing gradients and optimizer states for every weight — roughly 3–4× the model size in GPU memory. That's ~100 GB just to train. Not practical on a single GPU.

**LoRA** (Low-Rank Adaptation, Hu et al. 2021) sidesteps this by observing that the *change* needed to adapt a pre-trained model to a new task is low-rank. Instead of modifying the original weight matrix `W` directly, LoRA freezes `W` and adds a small bypass:

```
output = x @ W  +  x @ (A @ B)
              └─ frozen ─┘   └─ trained ─┘
```

- `A` has shape `(d_model, rank)` — initialized randomly  
- `B` has shape `(rank, d_model)` — initialized to zero (so training starts from the original model)
- `rank` is typically 8–64. At rank 16, a single attention weight goes from 4096×4096 = 16M params → 2×(4096×16) = 131K params — **120× fewer trainable parameters**

**QLoRA** (Dettmers et al. 2023) adds one more trick: the frozen base weights are stored in **4-bit quantization** (NF4 format), cutting memory by ~4×. The LoRA adapters themselves are kept in 16-bit for training precision. This is what lets a 7B model fit in ~10 GB VRAM.

The matrices that get LoRA adapters added are the **attention projections** (`q_proj`, `k_proj`, `v_proj`, `o_proj`) and optionally the **MLP feed-forward layers** (`gate_proj`, `up_proj`, `down_proj`). Adding MLP adapters gives more capacity but uses more VRAM.

## 2. Install dependencies

In [ ]:
# Unsloth wraps HuggingFace Transformers + PEFT with speed/memory optimisations.
# It also bundles trl (the SFTTrainer) and peft (the LoRA layer implementation).
#
# If this cell fails because of a CUDA version mismatch, visit:
#   https://github.com/unslothai/unsloth#installation
# and pick the pip install line matching your CUDA version.

!pip install --quiet unsloth datasets trl

In [ ]:
import torch

# Confirm a GPU is available and check how much VRAM you have.
# Fine-tuning Qwen2.5-Coder-7B with QLoRA needs ~10-12 GB.
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    vram_gb = gpu.total_memory / 1e9
    print(f"GPU  : {gpu.name}")
    print(f"VRAM : {vram_gb:.1f} GB")
    if vram_gb < 10:
        print("WARNING: less than 10 GB — reduce max_seq_length to 512 in step 4")
else:
    print("No GPU found. This notebook requires a CUDA GPU.")

## 3. Understand the training data

Before training anything, spend time understanding what you're training on. Every training example is a **conversation** between a user and assistant:

```
user:      "Optimize this C code..."   ← the prompt
assistant: "```c\n...optimized...\n```" ← what we want the model to learn to output
```

This format is called **supervised fine-tuning (SFT)** — we're showing the model correct (input, output) pairs and adjusting weights so it becomes more likely to produce those outputs given those inputs.

Let's look at what's actually in our JSONL file.

In [ ]:
import json

# ── Generate the JSONL file if it doesn't exist yet ──────────────────────────
# (Runs prepare_finetune_data.py from the same directory)
import subprocess, os
if not os.path.exists("train.jsonl"):
    print("Generating train.jsonl and val.jsonl from dataset/ ...")
    subprocess.run([
        "python3", "prepare_finetune_data.py",
        "--strategies", "generic", "pattern-aware",
        "--split", "0.9",
        "--train", "train.jsonl",
        "--val",   "val.jsonl",
    ], check=True)
else:
    print("train.jsonl already exists, skipping generation.")

In [ ]:
# Load all examples into memory so we can inspect them
with open("train.jsonl") as f:
    train_examples = [json.loads(line) for line in f]
with open("val.jsonl") as f:
    val_examples = [json.loads(line) for line in f]

print(f"Train examples : {len(train_examples)}")
print(f"Val   examples : {len(val_examples)}")
print()

# Print one example in full so you can see the exact format
example = train_examples[0]
print("─── messages ───────────────────────────────────────")
for msg in example["messages"]:
    role = msg["role"].upper()
    content = msg["content"]
    print(f"\n[{role}]\n{content[:600]}{'...' if len(content) > 600 else ''}")
print("\n────────────────────────────────────────────────────")

In [ ]:
# Check the distribution of token lengths.
# If the p95 is much larger than your max_seq_length, examples get truncated
# and training quality suffers. If it's much smaller, you can reduce
# max_seq_length to save VRAM.
#
# We use a rough heuristic: 1 token ≈ 4 characters for code.

lengths = [
    (len(ex["messages"][0]["content"]) + len(ex["messages"][1]["content"])) // 4
    for ex in train_examples
]
lengths.sort()
n = len(lengths)
print(f"Estimated token length per example:")
print(f"  min    : {lengths[0]}")
print(f"  median : {lengths[n//2]}")
print(f"  p95    : {lengths[int(n*0.95)]}")
print(f"  max    : {lengths[-1]}")
print()
print("Recommended max_seq_length: 1024 (covers most examples with headroom)")

## 4. Load the base model

`FastLanguageModel.from_pretrained` downloads Qwen2.5-Coder-7B-Instruct from HuggingFace (about 15 GB the first time) and loads it in 4-bit NF4 quantization.

**What `load_in_4bit` does:**  
Each weight that would normally be a 16-bit float is stored as a 4-bit integer using a special format called NF4 (Normal Float 4). The values are dequantized on-the-fly during the forward pass. This cuts VRAM from ~14 GB to ~4.5 GB for a 7B model, with minimal quality loss for inference. The LoRA adapters added in the next step are kept in full 16-bit precision.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 1024   # maximum tokens per training example (prompt + response)
                     # increase to 2048 if you have ≥16 GB VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "Qwen/Qwen2.5-Coder-7B-Instruct",
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,      # auto-detect: bfloat16 on Ampere+, float16 otherwise
    load_in_4bit   = True,      # QLoRA: store base weights in 4-bit NF4
)

# Verify: the tokenizer should have a chat template so it knows how to
# format user/assistant turns into the token sequence the model expects.
print("Tokenizer class  :", type(tokenizer).__name__)
print("Chat template set:", tokenizer.chat_template is not None)
print()

# Quick sanity check — apply the chat template to our first example
# and count tokens to confirm lengths are reasonable.
sample_tokens = tokenizer.apply_chat_template(
    train_examples[0]["messages"],
    tokenize=True,
    add_generation_prompt=False,
)
print(f"First example tokenised length: {len(sample_tokens)} tokens")

In [ ]:
# Let's see what the chat template actually produces.
# The model was trained to recognise special tokens like <|im_start|> and <|im_end|>
# to separate turns. Training examples must use exactly this format.

sample_text = tokenizer.apply_chat_template(
    train_examples[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,
)
print(sample_text[:800])

## 5. Apply LoRA adapters

`get_peft_model` injects the LoRA matrices into the model and freezes everything else. After this cell you can inspect exactly how many parameters are trainable vs frozen — this makes the memory savings concrete.

**Key hyperparameters to understand:**

| Parameter | What it controls | Typical range |
|-----------|-----------------|---------------|
| `r` (rank) | Capacity of the adapter. Higher = more expressive but uses more memory and can overfit | 8–64 |
| `lora_alpha` | Scales the adapter output: `output *= lora_alpha / r`. Setting `alpha = 2*r` is a common default | `2 * r` |
| `lora_dropout` | Dropout on the adapter path during training. Helps prevent overfitting on small datasets | 0–0.1 |
| `target_modules` | Which weight matrices get adapters. Adding MLP layers (`gate_proj` etc.) gives more capacity | attn only → attn+MLP |

In [ ]:
LORA_RANK = 16   # rank 16 is a good default: enough capacity, not too heavy

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,

    # These are all the linear layers in the attention and feed-forward blocks.
    # You can remove gate_proj/up_proj/down_proj to save VRAM at the cost of
    # slightly lower quality.
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",        # MLP
    ],

    lora_alpha     = LORA_RANK * 2,   # scale factor — keep at 2×rank
    lora_dropout   = 0.05,            # small dropout helps with a 700-example dataset
    bias           = "none",          # don't add bias terms to adapters
    use_gradient_checkpointing = "unsloth",  # trade compute for VRAM during backward pass
    random_state   = 42,
)

# Count trainable vs total parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:>12,}  ({100 * trainable / total:.2f}% of total)")
print(f"Frozen params    : {total - trainable:>12,}")
print(f"Total params     : {total:>12,}")
print()
print("Only the LoRA adapter weights (A and B matrices) are trained.")

## 6. Format data for training

The tokenizer's chat template converts our list of `{"role", "content"}` dicts into a single string with the model's special tokens. The SFTTrainer then tokenizes that string into input IDs.

One important detail: during training we only want to compute the loss on the **assistant's tokens**, not the user's prompt. If we computed loss on the prompt tokens too, we'd be penalising the model for not predicting the question it was given — which is unhelpful. SFTTrainer handles this automatically with response-only masking when we use the messages format.

In [ ]:
from datasets import load_dataset

# Load both splits from the JSONL files we generated
dataset = load_dataset("json", data_files={"train": "train.jsonl", "validation": "val.jsonl"})
print(dataset)
print()
print("First train example keys:", list(dataset["train"][0].keys()))

# Apply the chat template to convert messages → a single text string.
# This is what the trainer will tokenize.
def apply_template(batch):
    texts = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        for msgs in batch["messages"]
    ]
    return {"text": texts}

dataset = dataset.map(apply_template, batched=True, remove_columns=["messages"])
print("\nAfter applying template:")
print(dataset)
print()
print("── Sample text (first 600 chars) ──────────────────────────")
print(dataset["train"]["text"][0][:600])

## 7. Train

`SFTTrainer` is a thin wrapper around HuggingFace `Trainer` specialised for supervised fine-tuning. It handles tokenization, loss masking, gradient accumulation, and checkpointing.

**Training hyperparameters explained:**

| Hyperparameter | Effect | Our choice |
|---------------|--------|------------|
| `per_device_train_batch_size` | Examples processed per GPU per step. Limited by VRAM. | 2 |
| `gradient_accumulation_steps` | Simulates a larger batch by accumulating gradients before stepping. Effective batch = 2×8 = 16. | 8 |
| `num_train_epochs` | How many times to pass through the full dataset. More epochs = more learning but risk of overfitting. | 3 |
| `learning_rate` | Step size for the optimiser. LoRA adapters can tolerate higher LR than full fine-tuning. | 2e-4 |
| `warmup_ratio` | Fraction of steps to linearly ramp LR from 0. Prevents early instability. | 0.05 |
| `lr_scheduler_type` | How LR decays after warmup. Cosine is a smooth decay. | cosine |

**Watching the loss:** the training loss should decrease steadily over steps. If it plateaus early, try more epochs or a higher LR. If it oscillates wildly, lower the LR. If val loss starts increasing while train loss decreases — that's overfitting.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset  = dataset["validation"],
    args = SFTConfig(
        # ── Data ──────────────────────────────────────────────────────────
        dataset_text_field = "text",
        max_seq_length     = MAX_SEQ_LEN,

        # ── Batch / gradient ──────────────────────────────────────────────
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,     # effective batch size = 2 × 8 = 16

        # ── Schedule ──────────────────────────────────────────────────────
        num_train_epochs   = 3,
        learning_rate      = 2e-4,
        warmup_ratio       = 0.05,
        lr_scheduler_type  = "cosine",

        # ── Precision ─────────────────────────────────────────────────────
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),

        # ── Logging / saving ──────────────────────────────────────────────
        logging_steps      = 10,   # print loss every 10 steps
        eval_strategy      = "epoch",
        save_strategy      = "epoch",
        save_total_limit   = 2,    # keep only the 2 most recent checkpoints
        load_best_model_at_end = True,
        metric_for_best_model  = "eval_loss",
        output_dir         = "checkpoints/",
        report_to          = "none",   # change to "wandb" if you want experiment tracking
    ),
)

# Show an estimate of total training steps before we commit to running
steps_per_epoch = len(dataset["train"]) // (2 * 8)   # batch * grad_accum
total_steps     = steps_per_epoch * 3
print(f"Training examples : {len(dataset['train'])}")
print(f"Steps per epoch   : {steps_per_epoch}")
print(f"Total steps       : {total_steps}")

In [ ]:
# ── Run training ─────────────────────────────────────────────────────────────
# On an RTX 4090 (~24 GB VRAM) this takes roughly 10-15 minutes for 3 epochs.
# On an A100 (~40 GB VRAM) it takes ~5-8 minutes.
#
# You will see log lines like:
#   {'loss': 1.2345, 'learning_rate': 0.0002, 'epoch': 0.5}
# Watch the loss decrease. A good run typically goes from ~1.5 → ~0.4.

trainer_stats = trainer.train()

print(f"\nTraining complete.")
print(f"  Total steps   : {trainer_stats.global_step}")
print(f"  Training time : {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"  Final loss    : {trainer_stats.metrics['train_loss']:.4f}")

## 8. Inspect the loss curve

The training log is stored in `trainer.state.log_history`. Plotting it helps you see whether training converged, plateaued early, or overfit.

In [ ]:
import matplotlib.pyplot as plt

log = trainer.state.log_history

train_steps  = [e["step"] for e in log if "loss" in e]
train_losses = [e["loss"] for e in log if "loss" in e]
val_steps    = [e["step"] for e in log if "eval_loss" in e]
val_losses   = [e["eval_loss"] for e in log if "eval_loss" in e]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_steps, train_losses, label="Train loss",      color="steelblue")
ax.plot(val_steps,   val_losses,   label="Validation loss", color="tomato", marker="o")
ax.set_xlabel("Step")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Training curve — Qwen2.5-Coder-7B QLoRA")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("loss_curve.png", dpi=120)
plt.show()

# What to look for:
# ✓ Both curves decrease — training is working
# ✓ Val loss tracks train loss closely — no overfitting
# ✗ Val loss rises while train continues to fall — overfitting
#   → Fix: fewer epochs, higher dropout, or more training data
# ✗ Both curves plateau after epoch 1 — underfitting
#   → Fix: higher learning rate, more epochs, or higher rank

## 9. Benchmark the fine-tuned model on the full dataset

Now that training is complete, evaluate the fine-tuned model against every variant in `../dataset/`. For each variant the notebook will:

1. Read `slow.c` and build a generic optimization prompt
2. Run inference with the fine-tuned model (in-memory, no Ollama needed)
3. Extract the C code from the model's response
4. Compile the output as a separate translation unit alongside `slow.c` and `test.c`
5. Measure the speedup vs the slow baseline

This mirrors exactly how `evaluate_llm.py --dataset ../dataset/` works, but drives the model directly rather than going through an API. All 590 variants with test harnesses are evaluated; the per-pattern summary at the end shows which patterns the model successfully optimized.

**Expected runtime:** ~5–20 minutes depending on GPU and number of variants.

In [ ]:
import os, re, json, sys
from pathlib import Path
from collections import defaultdict

# REPO_ROOT and sys.path were set in cell 0 (Colab setup).
# If you skipped cell 0, uncomment and set this manually:
# import sys; sys.path.insert(0, '/content/benchmark')

REPO_ROOT = Path(sys.path[0])   # first entry added by cell 0

from compiler import compile_and_run_variant, extract_code_from_response

_PORTABILITY_NOTE = (
    "Important: write portable standard C (C99/C11). "
    "Do NOT use x86-specific intrinsics (SSE/AVX/AVX2) or any "
    "#include <immintrin.h> / <xmmintrin.h> / <emmintrin.h>. "
    "Use plain loops that the compiler can auto-vectorize."
)


def _rename_to(code: str, target_name: str) -> str:
    """Rename the last function definition in *code* to *target_name*."""
    matches = list(re.finditer(r'\b([A-Za-z_]\w*)\s*\([^)]*\)\s*\{', code, re.MULTILINE))
    func_name = None
    for m in reversed(matches):
        name = m.group(1)
        if name not in ("if", "for", "while", "switch", "return"):
            func_name = name
            break
    if func_name is None or func_name == target_name:
        return code
    return re.sub(r'\b' + re.escape(func_name) + r'\b', target_name, code)


def build_generic_prompt(slow_code: str) -> str:
    return (
        "Optimize the following C function for performance. "
        "Identify and eliminate any inefficiencies such as redundant computation, "
        "unnecessary memory accesses, or suboptimal algorithms.\n"
        "Rename the function to `optimized`. "
        "Return ONLY the optimized C function — no explanation, no comments about what changed.\n"
        f"{_PORTABILITY_NOTE}\n\n"
        f"```c\n{slow_code}\n```"
    )


def load_variants(dataset_dir: Path):
    """Yield variant dicts for every variant that has a test harness."""
    for root, dirs, files in sorted(os.walk(dataset_dir)):
        if "metadata.json" not in files or "slow.c" not in files:
            continue
        test_path = os.path.join(root, "test.c")
        if not os.path.exists(test_path):
            continue
        with open(test_path) as f:
            test_harness = f.read()
        if "SLOW_CODE_HERE" not in test_harness:
            continue
        with open(os.path.join(root, "metadata.json")) as f:
            meta = json.load(f)
        if meta.get("compiler_fixable", False):
            continue
        with open(os.path.join(root, "slow.c")) as f:
            slow_code = f.read()
        with open(os.path.join(root, "fast.c")) as f:
            fast_code = f.read()
        helper = os.path.join(root, "helper.c")
        yield {
            "variant_id":   meta["variant_id"],
            "pattern_id":   meta["pattern_id"],
            "metadata":     meta,
            "slow_code":    slow_code,
            "fast_code":    fast_code,
            "test_harness": test_harness,
            "helper_path":  helper if os.path.exists(helper) else None,
        }


DATASET_DIR = REPO_ROOT / "dataset"
variants = list(load_variants(DATASET_DIR))
print(f"dataset  : {DATASET_DIR}")
print(f"variants : {len(variants)}")

In [ ]:
# ── Inference + compile + benchmark for every variant ─────────────────────────
# Switch the model to fast inference mode (Unsloth's optimised kernels).
FastLanguageModel.for_inference(model)

eval_results = []
n_total = len(variants)

for i, v in enumerate(variants, 1):
    variant_id   = v["variant_id"]
    pattern_id   = v["pattern_id"]
    slow_code    = v["slow_code"]
    test_harness = v["test_harness"]
    helper_path  = v["helper_path"]

    # The test harness calls fast_<suffix>; discover the exact name.
    m = re.search(r'\bfast_\w+', test_harness)
    fast_func_name = m.group(0) if m else "optimized"

    # ── Generate ──────────────────────────────────────────────────────────────
    prompt = build_generic_prompt(slow_code)
    input_ids = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens  = 512,
            temperature     = 0.1,
            do_sample       = True,
            pad_token_id    = tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        output_ids[0][input_ids.shape[1]:], skip_special_tokens=True
    )

    # Extract C code and rename the last function to what test.c expects.
    llm_code = extract_code_from_response(response)   # normalises → "optimized"
    llm_code = _rename_to(llm_code, fast_func_name)   # "optimized" → "fast_xxx"

    # ── Compile + run ─────────────────────────────────────────────────────────
    result   = compile_and_run_variant(
        llm_code, slow_code, test_harness,
        helper_path=helper_path, runs=3,
    )

    compiles = result.get("compiles", False)
    correct  = result.get("correct",  False)
    slow_ms  = result.get("slow_ms",  0.0)
    llm_ms   = result.get("fast_ms",  0.0)
    speedup  = slow_ms / llm_ms if llm_ms > 0 and slow_ms > 0 else 0.0

    eval_results.append({
        "variant_id": variant_id,
        "pattern_id": pattern_id,
        "compiles":   compiles,
        "correct":    correct,
        "slow_ms":    slow_ms,
        "llm_ms":     llm_ms,
        "speedup":    speedup,
        "llm_code":   llm_code,
    })

    status  = "✓" if correct else ("✗ compile" if not compiles else "✗ wrong")
    spd_str = f"  {speedup:.2f}x" if correct else ""
    print(f"[{i:>4}/{n_total}] {variant_id:<22} {status}{spd_str}")

n_correct = sum(r["correct"] for r in eval_results)
print(f"\n{'='*60}")
print(f"  {n_correct}/{n_total} correct  ({100*n_correct/n_total:.1f}%)")
print(f"{'='*60}")

In [ ]:
# ── Per-pattern summary ────────────────────────────────────────────────────────
by_pattern = defaultdict(list)
for r in eval_results:
    by_pattern[r["pattern_id"]].append(r)

header = f"{'Pattern':<10} {'Variants':>8} {'Correct':>8} {'Avg speedup':>12}  {'≥2x':>5}"
print(header)
print("─" * len(header))

for pid in sorted(by_pattern.keys()):
    rows   = by_pattern[pid]
    n      = len(rows)
    c_rows = [r for r in rows if r["correct"]]
    n_corr = len(c_rows)
    avg_spd = sum(r["speedup"] for r in c_rows) / n_corr if n_corr else 0.0
    at_2x   = sum(1 for r in c_rows if r["speedup"] >= 2.0)
    print(f"{pid:<10} {n:>8} {n_corr:>8} {avg_spd:>11.2f}x  {at_2x}/{n_corr}")

print("─" * len(header))
n_total_correct = sum(r["correct"] for r in eval_results)
n_total_2x      = sum(r["speedup"] >= 2.0 for r in eval_results if r["correct"])
print(f"{'TOTAL':<10} {len(eval_results):>8} {n_total_correct:>8}  "
      f"{n_total_2x}/{n_total_correct} variants with ≥2x speedup")

## 10. Save the LoRA adapter and serve it

You have two options for deploying the model:

**Option A — save the adapter only** (small, ~100 MB)  
The base model is not modified. You load both at inference time. Good for quick experimentation.

**Option B — merge weights and export to GGUF** (for Ollama)  
Merges the LoRA deltas into the base weights permanently, then converts to GGUF format which Ollama can serve. This is what you need for `evaluate_llm.py` to use it.

In [ ]:
# ── Option A: Save just the LoRA adapter (~100 MB) ───────────────────────────
# Use this to reload quickly in Python without re-downloading the base model.
#
#   model, tokenizer = FastLanguageModel.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct", ...)
#   model.load_adapter("lora_adapter/")

model.save_pretrained("lora_adapter/")
tokenizer.save_pretrained("lora_adapter/")
print("LoRA adapter saved to lora_adapter/")

import os
adapter_files = os.listdir("lora_adapter/")
print(f"Files: {adapter_files}")
# adapter_model.safetensors  — the actual A and B matrices (~100 MB)
# adapter_config.json        — rank, alpha, target_modules etc.
# tokenizer files            — same tokenizer, included for convenience

In [ ]:
# ── Option B: Merge weights and export to GGUF (for Ollama) ──────────────────
# This merges the LoRA deltas permanently into the base weights, then
# quantizes to Q4_K_M GGUF format. The resulting file is ~4.5 GB.
#
# Unsloth handles the full pipeline — no need for a separate llama.cpp step.

model.save_pretrained_gguf(
    "gguf_export/",
    tokenizer,
    quantization_method = "q4_k_m",   # Q4_K_M: good quality/size tradeoff
                                       # Other options: q8_0 (higher quality, 8 GB)
                                       #                f16  (no quant, 14 GB)
)

# After this completes you will have:
#   gguf_export/unsloth.Q4_K_M.gguf   (~4.5 GB)

print("\nNext: register with Ollama")
print("  echo 'FROM ./gguf_export/unsloth.Q4_K_M.gguf' > Modelfile")
print("  ollama create qwen2.5-coder-finetuned -f Modelfile")
print("  ollama run qwen2.5-coder-finetuned")
print()
print("Then add to models.yaml and run the benchmark comparison:")
print("  python3 evaluate_llm.py --model qwen2.5-coder-finetuned --strategy generic")

---

## What to experiment with next

Now that you have a working fine-tuning pipeline, here are the most impactful things to try:

**More training signal**
- Add the third prompt strategy: `--strategies generic pattern-aware taxonomy-guided` — triples examples to ~1770 for free
- Generate additional variants per pattern (edit `N_VARIANTS` in `generate_variants.py`)

**Overfitting test**  
Hold out all variants of one entire pattern (e.g. remove all `AL-1` from training). After fine-tuning, check if the model generalises to `AL-1` at evaluation. A model that only memorised training examples will fail this test.

**Rank ablation**  
Train three runs with `r=8`, `r=16`, `r=32`. Compare val loss and eval benchmark score. More rank doesn't always win.

**Longer training**  
Try `num_train_epochs=5` and watch the loss curve. If val loss starts increasing, you've found the overfitting point for your dataset size.

**Prompt strategy matters**  
Compare evaluation scores from the model when prompted with `generic` vs `taxonomy-guided`. A model fine-tuned on `generic` prompts may be worse when given extra hints — or better, because fine-tuning shifted its priors.